# Step 2 — Feature engineering and non-learned baselines

Builds ~105 features for every displayed (user, item) pair and evaluates simple one-signal baselines on the same full slates every model is evaluated on.

### Setup (run first)

**On Google Colab:** *Runtime → Change runtime type → GPU* (T4 is enough; A100/L4 is faster). Put the two raw JDsearch
files in `MyDrive/bt4222/raw/` (`user_behavior_data.txt`, `product_meta_data.txt`). The cell mounts Drive, clones the
repository, installs the requirements and writes every output to `MyDrive/bt4222/outputs/`, so results persist between
sessions and the notebooks can be run one after another (or on different days).

**Locally:** run the notebook from the repository with the virtual environment active; data is read from `data/raw/`
and outputs go to `outputs/`.

`CONFIG = "configs/smoke.yaml"` runs everything on a 2,000-user sample with tiny models (minutes, CPU) to check the code;
`configs/default.yaml` is the real run. Every stage skips work whose outputs already exist; pass `"--force"` to redo one.

In [1]:
import os, subprocess, sys
from pathlib import Path

CONFIG = "configs/default.yaml"          # "configs/smoke.yaml" for a quick check
REPO_URL = "https://github.com/clarequek/bt4222-two-tower-recsys.git"
DRIVE_DIR = "/content/drive/MyDrive/bt4222"   # Colab: raw/ (input) and outputs/ (written by the pipeline)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/bt4222-two-tower-recsys")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

from jds.notebook import Runner, banner
run = Runner(CONFIG, drive_dir=DRIVE_DIR if IN_COLAB else None)

config configs/default.yaml | scope full | raw data /home/ubuntu/bt4222-two-tower-recsys/data/raw | outputs /home/ubuntu/bt4222-two-tower-recsys/outputs/full


## 2.1 Features (`scripts/02_features.py`, `src/jds/features.py`)

Each data field has one role:

| data field | role |
|---|---|
| test query + displayed slate + labels | the supervised task (query-conditioned re-ranking of the slate) |
| the user's own history | personal features, **counted per event type inside that type's retention window** (`click_7`, `cart_90`, `order_365`, …) plus days since the last event |
| query-tagged history of all users | query → item / brand / category engagement (`qpop_*`) |
| history of all users | windowed item / attribute popularity (`ipop_*`, `apop_*`) |
| product metadata | BM25 and query-term coverage (`lex_*`), item content — needed because most displayed items are cold |
| labels of *other* training users | impression CTR (`ctr_*`), **out-of-fold**: a training user never sees its own labels |

Also: engagement under the user's earlier queries similar to the test query (`u_q_*`, `u_simq_*`), carted-but-not-ordered
(`u_cart_not_ordered`), slate context and within-slate percentiles (`slate_*`, `*_pct`).
Peak RAM is ~6 GB; on a small Colab runtime add `"--set", "features.chunk_users=10000"`.

### Key functions (`src/jds/features.py`)

* **`build`** drives the stage. It computes the global tables once from all users, then processes users in chunks (`features.chunk_users`) to bound memory. Label-based features are joined first; `list_context` runs last because its percentiles need the final, null-filled columns.
* **`popularity_tables`** counts events per item, attribute and (query, item) over all users' history, inside each type's window. No labels are used.
* **`ctr_tables`** computes smoothed rates from the labelled impressions of a set of users:
  `ctr = (positives + prior × global_rate) / (impressions + prior)`, with `prior = 50`. Shrinkage stops rarely displayed items from getting extreme rates that a ranker could memorise. It also produces `gain_*` (summed label / (impressions + prior), the graded version) and `disp_*` (impressions per 1,000 users, which stays comparable between tables built from different numbers of users).
* **`oof_ctr_tables`** and **`join_ctr`** prevent label leakage. A training user in fold *f* takes the table built from the other folds' users; validation and test users take the table built from all training users. Keys unseen in a table fall back to the global rate.
* **`personal_features`** matches the user's own history to the candidate (same item, brand, shop, category) per type window, adds days since the last event, and sets `u_cart_not_ordered` for an item carted in the window and not ordered since (an open purchase intention).
* **`query_history_features`** finds the user's earlier queries with term Jaccard ≥ 0.5 against the test query, then counts engagement under them. Similarity is needed because the exact test query occurs in only 1 of 173,831 users' histories.
* **`term_idf`** and **`lexical`** score the test query against the item text with BM25 (`k1 = 1.2`, `b = 0.75`, idf estimated on a sample of items) and add query-term coverage of the full text and of the name. These signals work for cold items.
* **`list_context`** describes each candidate relative to its slate: the share of the slate with the same brand, shop or category, and within-slate percentile ranks. A ranker only has to order items inside a slate, so relative values are often more useful than raw ones.

In [2]:
run.stage("02_features.py")

$ /home/ubuntu/bt4222-two-tower-recsys/.venv/bin/python -u scripts/02_features.py --config configs/default.yaml


06:02:09 features | features exist in /home/ubuntu/bt4222-two-tower-recsys/outputs/full/features (use --force to rebuild)


105 features


In [3]:
import json
names = json.loads((run.p.features / "feature_list.json").read_text())
print(len(names), "features"); names[:20]

105 features


['ctr_wid',
 'gain_wid',
 'disp_wid',
 'ctr_brand_id',
 'gain_brand_id',
 'disp_brand_id',
 'ctr_shop_id',
 'gain_shop_id',
 'disp_shop_id',
 'ctr_cate_id_4',
 'gain_cate_id_4',
 'disp_cate_id_4',
 'ctr_q_wid',
 'gain_q_wid',
 'disp_q_wid',
 'ctr_q_cate_id_4',
 'gain_q_cate_id_4',
 'disp_q_cate_id_4',
 'ctr_q_brand_id',
 'gain_q_brand_id']

## 2.2 Baselines (`scripts/02b_baselines.py`, `src/jds/baselines.py`)

Each baseline ranks a slate by one signal (random, windowed popularity, CTR, query–item engagement, BM25, user affinity),
plus a popularity + BM25 mix whose weight is picked on validation users. Metrics: MRR, P@1, NDCG@10 (binary, as in the
JDsearch paper), graded NDCG@10 (gain 2^label − 1), HR@10, recall@10/50.

### Key functions

* **`baselines.run`** (`src/jds/baselines.py`) ranks each slate by one feature column. The ensemble score is `pop_pct + w × bm25_pct`, two within-slate percentiles so the signals share one scale. `w` comes from `baselines.ensemble_grid` and is the value with the best graded NDCG@10 on validation users, so test users play no part in choosing it.
* **`metrics.add_ranks`** (`src/jds/metrics.py`) ranks by score and breaks ties with a seeded random key, never by list position, because the raw order leaks the label.
* **`metrics.per_user_metrics`** computes each metric per user and drops users without a positive; **`metrics.evaluate`** averages over users, so every user counts equally whatever the slate size.

In [4]:
run.stage("02b_baselines.py")

$ /home/ubuntu/bt4222-two-tower-recsys/.venv/bin/python -u scripts/02b_baselines.py --config configs/default.yaml


06:02:14 baselines | ensemble weight on BM25 picked on validation: 0.0 (gNDCG@10 by weight: {0.0: np.float64(0.3329), 0.1: np.float64(0.3299), 0.25: np.float64(0.3054), 0.5: np.float64(0.2745), 1.0: np.float64(0.2245), 2.0: np.float64(0.1723)})


06:02:15 baselines | 


shape: (11, 9)


┌──────────────────────────────┬────────┬────────┬────────┬───┬────────┬────────┬────────┬─────────┐


...

│ User affinity: same category ┆ 0.1106 ┆ 0.0353 ┆ 0.0969 ┆ … ┆ 0.2508 ┆ 0.1813 ┆ 0.65   ┆ 17174   │


│ c…                           ┆        ┆        ┆        ┆   ┆        ┆        ┆        ┆         │


│ Popularity + 0.0 x BM25      ┆ 0.3311 ┆ 0.1915 ┆ 0.3386 ┆ … ┆ 0.6349 ┆ 0.5207 ┆ 0.8403 ┆ 17174   │


│ [test]                       ┆        ┆        ┆        ┆   ┆        ┆        ┆        ┆         │


└──────────────────────────────┴────────┴────────┴────────┴───┴────────┴────────┴────────┴─────────┘


In [5]:
banner()
import pandas as pd
b = pd.DataFrame(run.json("baselines.json"))
b = b[b.model.str.endswith("[test]")]
b.style.format(precision=4).highlight_max(subset=["mrr", "p1", "ndcg10", "gndcg10", "hr10"], color="#c6efce")

,model,mrr,p1,ndcg10,gndcg10,hr10,r10,r50,n_users
10,Random [test],0.1005,0.0298,0.0850,0.0833,0.2266,0.1599,0.6081,17174
11,Popularity: users who interacted (all history) [test],0.3435,0.2012,0.3525,0.3479,0.6526,0.5403,0.8660,17174
12,"Popularity: clicks, last 7 days [test]",0.3311,0.1915,0.3386,0.3340,0.6349,0.5207,0.8403,17174
13,"Popularity: carts, last 90 days [test]",0.3344,0.1952,0.3424,0.3382,0.6353,0.5237,0.8459,17174
14,"Popularity: orders, last 365 days [test]",0.2929,0.1649,0.2954,0.2914,0.5646,0.4578,0.7942,17174
15,Impression CTR of item (training users) [test],0.2800,0.1705,0.2718,0.2681,0.5027,0.3985,0.6327,17174
16,Query-item CTR (training users) [test],0.1901,0.1073,0.1730,0.1701,0.3396,0.2579,0.5932,17174
17,"Query-item history engagement (clicks, 7 days) [test]",0.2441,0.1399,0.2343,0.2310,0.4461,0.3520,0.7084,17174
18,BM25 (query vs item text) [test],0.1096,0.0385,0.0959,0.0938,0.2352,0.1734,0.6327,17174
19,"User affinity: same category clicked, 7 days [test]",0.1106,0.0353,0.0969,0.0950,0.2508,0.1813,0.6500,17174
